<a href="https://colab.research.google.com/github/Ialdanah1/agentic-ai-engineering-labs/blob/main/01-langgraph-agent-observability/agent_execution_logging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Agent Execution Logging </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to implement **Agent Execution Logging** in LangGraph-based agent systems — capturing execution traces, logging tool calls, tracking reasoning steps, and measuring cost & latency.

**Scenario:** The AI Detective 🕵️  
A LangGraph agent investigates the theft of the Rosewood Emerald from Rosewood Manor — checking suspects' alibis, searching crime scenes for evidence, and calculating case metrics to identify the culprit — while a custom observability layer silently records every decision, tool invocation, and reasoning chain for debugging and auditing.

**Key Topics Covered:**
- Logging agent decisions and reasoning steps
- Recording tool calls with inputs / outputs
- Building structured execution traces
- Tracking latency per step and total cost estimation

---


## 1. Setup

In [1]:
# Run this cell first
!pip install -q langchain langchain-openai langgraph langchain-community --quiet

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Enter your OpenAI API key: ··········
Setup complete


In [2]:
import time
import uuid
import json
import logging
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Any, Optional

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM and libraries ready")


LLM and libraries ready


## 2. Observability Layer — Core Infrastructure

Before defining any tools or agents we build the **observability backbone**:

| Component | Purpose |
|---|---|
| `ExecutionTrace` | Structured record for a single agent run |
| `StepLog` | One logged event (tool call, decision, reasoning) |
| `ObservabilityLogger` | Singleton that stores all traces and prints reports |

This mirrors production systems like LangSmith or Weights & Biases, but runs entirely in-process so you can see exactly what is captured.


In [3]:
# Data structures

from dataclasses import dataclass, field

@dataclass
class StepLog:
    """A single logged step inside an agent run."""
    step_id:    str
    step_type:  str          # 'reasoning' | 'tool_call' | 'tool_result' | 'decision'
    timestamp:  str
    content:    Any
    latency_ms: float = 0.0
    metadata:   Dict[str, Any] = field(default_factory=dict)


@dataclass
class ExecutionTrace:
    """Full trace for one agent invocation."""
    trace_id:       str
    query:          str
    started_at:     str
    steps:          List[StepLog] = field(default_factory=list)
    finished_at:    Optional[str] = None
    total_ms:       float = 0.0
    estimated_cost: float = 0.0   # USD — rough estimate
    final_answer:   Optional[str] = None
    status:         str = "running"   # running | success | error


class ObservabilityLogger:
    """Central store for all execution traces."""

    def __init__(self):
        self.traces: Dict[str, ExecutionTrace] = {}
        self._current_trace_id: Optional[str] = None

    # Trace lifecycle

    def start_trace(self, query: str) -> str:
        trace_id = str(uuid.uuid4())[:8]
        self.traces[trace_id] = ExecutionTrace(
            trace_id=trace_id,
            query=query,
            started_at=datetime.utcnow().isoformat(),
        )
        self._current_trace_id = trace_id
        print(f"\n{'='*65}")
        print(f"    NEW TRACE STARTED  [{trace_id}]")
        print(f"  Query: {query[:80]}{'...' if len(query)>80 else ''}")
        print(f"{'='*65}")
        return trace_id

    def end_trace(self, trace_id: str, final_answer: str,
                  status: str = "success", total_tokens: int = 0):
        trace = self.traces[trace_id]
        trace.finished_at = datetime.utcnow().isoformat()
        start = datetime.fromisoformat(trace.started_at)
        end   = datetime.fromisoformat(trace.finished_at)
        trace.total_ms = (end - start).total_seconds() * 1000
        trace.final_answer = final_answer
        trace.status = status
        # gpt-4o-mini pricing: ~$0.15/1M input tokens (rough estimate)
        trace.estimated_cost = (total_tokens / 1_000_000) * 0.15
        print(f"\n{'='*65}")
        print(f"  TRACE COMPLETE  [{trace_id}]  status={status}")
        print(f"  Total time : {trace.total_ms:.0f} ms")
        print(f"  Est. cost  : ${trace.estimated_cost:.6f}")
        print(f"{'='*65}\n")

    #  Step logging

    def log_step(self, trace_id: str, step_type: str,
                 content: Any, latency_ms: float = 0.0,
                 metadata: Dict[str, Any] = None) -> StepLog:
        step = StepLog(
            step_id   = str(uuid.uuid4())[:6],
            step_type = step_type,
            timestamp = datetime.utcnow().isoformat(),
            content   = content,
            latency_ms= latency_ms,
            metadata  = metadata or {},
        )
        self.traces[trace_id].steps.append(step)

        # Pretty console output
        icons = {
            "reasoning":   "🧠",
            "tool_call":   "🔧",
            "tool_result": "📦",
            "decision":    "⚡",
        }
        icon = icons.get(step_type, "•")
        content_preview = str(content)[:120]
        print(f"  {icon} [{step_type.upper():12s}] {content_preview}")
        if latency_ms:
            print(f"              ⏱  {latency_ms:.0f} ms")
        return step

    # Reporting

    def print_full_report(self, trace_id: str):
        trace = self.traces[trace_id]
        print(f"\n{'#'*65}")
        print(f"  EXECUTION TRACE REPORT — [{trace_id}]")
        print(f"{'#'*65}")
        print(f"  Query        : {trace.query}")
        print(f"  Status       : {trace.status}")
        print(f"  Started      : {trace.started_at}")
        print(f"  Finished     : {trace.finished_at}")
        print(f"  Total time   : {trace.total_ms:.0f} ms")
        print(f"  Est. cost    : ${trace.estimated_cost:.6f}")
        print(f"  Steps logged : {len(trace.steps)}")
        print()

        for i, step in enumerate(trace.steps, 1):
            print(f"  Step {i:02d} | {step.step_type.upper():12s} | "
                  f"{step.latency_ms:6.0f} ms | {str(step.content)[:90]}")

        print()
        print(f"  Final Answer : {str(trace.final_answer)[:200]}")
        print(f"{'#'*65}\n")

    def print_summary_dashboard(self):
        """Print a high-level summary across all traces."""
        print(f"\n{'='*65}")
        print("  📊  OBSERVABILITY DASHBOARD — ALL TRACES")
        print(f"{'='*65}")
        print(f"  {'Trace':8s} {'Status':8s} {'Steps':6s} "
              f"{'Time(ms)':10s} {'Est.Cost':10s}  Query")
        print(f"  {'-'*60}")
        for t in self.traces.values():
            query_short = t.query[:35] + "..." if len(t.query) > 35 else t.query
            print(f"  {t.trace_id:8s} {t.status:8s} {len(t.steps):6d} "
                  f"{t.total_ms:10.0f} ${t.estimated_cost:.6f}  {query_short}")
        print(f"{'='*65}\n")


# Instantiate the global logger
obs_logger = ObservabilityLogger()

print("Observability logger initialised ")


Observability logger initialised 


## 3. Instrumented Detective Tools

Each tool is wrapped with **timing** and **structured logging** so every call appears in the execution trace automatically.  
Three tools simulate a real detective-investigation stack:

| Tool | What it does |
|---|---|
| `check_alibi` | Checks a suspect's alibi for the night of the theft |
| `search_crime_scene_evidence` | Searches a location in the manor for physical evidence |
| `calculate_case_metric` | Computes guilt score, case strength, or innocence probability |


In [4]:
import random

# Mock case data — The Vanishing Emerald of Rosewood Manor
MOCK_ALIBIS = {
    "CHEF_DUBOIS": {"alibi": "Claims he was preparing dinner in the kitchen all evening.", "verified": False},
    "BUTLER_GREY": {"alibi": "Says he was polishing silverware in the dining room.", "verified": True},
    "LADY_SINHA": {"alibi": "States she was reading in the west wing library.", "verified": False},
    "GARDENER_HAYES": {"alibi": "Insists he was locking up the greenhouse at the time.", "verified": True},
    "MS_WARD": {"alibi": "Says she was reviewing the manor's accounts in her study.", "verified": False},
}

MOCK_EVIDENCE = {
    "STUDY": [
        "A muddy footprint near the safe.",
        "A torn piece of green fabric caught on the window latch.",
        "The safe's lock shows fresh scratch marks.",
    ],
    "GREENHOUSE": [
        "Fresh soil disturbed near the potting shed.",
        "A pair of gardening gloves, oddly clean.",
        "No sign of forced entry.",
    ],
    "GALLERY": [
        "Faint fingerprints on the display case glass.",
        "A guest book missing its last page.",
        "Security camera unplugged at 9:47 PM.",
    ],
    "KITCHEN": [
        "Knife set slightly out of order.",
        "A receipt for a locksmith, dated last week.",
    ],
}

#  Tool definitions

@tool
def check_alibi(suspect: str) -> str:
    """Check a suspect's alibi for the night the Rosewood Emerald was stolen."""
    time.sleep(0.05)   # simulate records-office lookup latency
    suspect = suspect.upper().replace(" ", "_")
    if suspect in MOCK_ALIBIS:
        d = MOCK_ALIBIS[suspect]
        status = "✅ Verified" if d["verified"] else "⚠️ Unverified"
        return f"{suspect}: \"{d['alibi']}\"  {status}"
    return f"No alibi on file for '{suspect}'."


@tool
def search_crime_scene_evidence(location: str) -> str:
    """Search a location in Rosewood Manor for physical evidence."""
    time.sleep(0.08)
    location = location.upper()
    findings = MOCK_EVIDENCE.get(location, [f"No evidence catalogued for {location}."])
    return "\n".join(f"• {f}" for f in findings)


@tool
def calculate_case_metric(
    metric: str,
    evidence_count: float,
    alibi_strength: float = 1.0,
    suspects_interviewed_count: float = 5.0,
) -> str:
    """
    Calculate an investigation metric.

    metric: 'guilt_score' | 'case_strength' | 'innocence_probability'
    evidence_count: number of pieces of evidence tying a suspect to the scene
    alibi_strength: how solid the suspect's alibi is (0-1, higher = stronger alibi)
    suspects_interviewed_count: number of suspects interviewed so far
    """
    time.sleep(0.03)
    metric = metric.lower()
    if metric == "guilt_score":
        if alibi_strength <= 0:
            return "Cannot calculate guilt score: alibi strength must be positive."
        return f"Guilt Score = {evidence_count / alibi_strength:.2f}"
    elif metric == "case_strength":
        strength = evidence_count * suspects_interviewed_count
        return f"Case Strength = {strength:,.0f} points  ({strength/100:.1f}% case file complete)"
    elif metric == "innocence_probability":
        if evidence_count <= 0:
            return "Cannot calculate: evidence count must be positive."
        return f"Innocence Probability = {(alibi_strength / evidence_count) * 100:.2f}%"
    return f"Unknown metric '{metric}'. Choose: guilt_score, case_strength, innocence_probability."


tools = [check_alibi, search_crime_scene_evidence, calculate_case_metric]

print(f"{len(tools)} instrumented detective tools ready ")


3 instrumented detective tools ready 


## 4. LangGraph Agent with Observability Hooks

We build a **LangGraph ReAct-style agent** and inject observability hooks at each node:

```
START → [agent_node] → [tool_node] → [agent_node] → ... → END
                ↑               ↑
          logs reasoning    logs tool I/O
```

The agent state carries a `trace_id` so every log event is linked to the right trace.


In [5]:
from langgraph.graph.message import add_messages
from typing import Annotated

#  State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    trace_id: str          # passed through the entire graph
    step_count: int
    total_tokens: int

#  Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Agent node
def agent_node(state: AgentState) -> AgentState:
    """Core reasoning node — calls the LLM and logs the decision."""
    trace_id   = state["trace_id"]
    step_count = state["step_count"] + 1
    t0 = time.perf_counter()

    response = llm_with_tools.invoke(state["messages"])
    latency  = (time.perf_counter() - t0) * 1000

    # Estimate token usage
    usage = getattr(response, "usage_metadata", None)
    new_tokens = 0
    if usage:
        new_tokens = usage.get("total_tokens", 0)
    total_tokens = state["total_tokens"] + new_tokens

    #  Log reasoning
    if response.content:
        obs_logger.log_step(
            trace_id   = trace_id,
            step_type  = "reasoning",
            content    = response.content,
            latency_ms = latency,
            metadata   = {"step": step_count, "tokens": new_tokens},
        )

    # Log tool call decisions
    if hasattr(response, "tool_calls") and response.tool_calls:
        for tc in response.tool_calls:
            obs_logger.log_step(
                trace_id  = trace_id,
                step_type = "decision",
                content   = f"Calling tool '{tc['name']}' with args {tc['args']}",
                metadata  = {"tool_name": tc["name"], "tool_args": tc["args"]},
            )

    return {
        "messages":    [response],
        "trace_id":    trace_id,
        "step_count":  step_count,
        "total_tokens": total_tokens,
    }


# Observable tool node
base_tool_node = ToolNode(tools)

def observable_tool_node(state: AgentState) -> AgentState:
    """Wraps ToolNode to log every tool invocation and its result."""
    trace_id = state["trace_id"]
    last_msg = state["messages"][-1]

    # Log each tool call before execution
    if hasattr(last_msg, "tool_calls"):
        for tc in last_msg.tool_calls:
            obs_logger.log_step(
                trace_id  = trace_id,
                step_type = "tool_call",
                content   = f"{tc['name']}({tc['args']})",
                metadata  = {"tool_name": tc["name"]},
            )

    t0 = time.perf_counter()
    result = base_tool_node.invoke(state)
    latency = (time.perf_counter() - t0) * 1000

    # Log results
    for msg in result.get("messages", []):
        if isinstance(msg, ToolMessage):
            obs_logger.log_step(
                trace_id   = trace_id,
                step_type  = "tool_result",
                content    = msg.content,
                latency_ms = latency,
                metadata   = {"tool_call_id": msg.tool_call_id},
            )

    return {**result, "trace_id": trace_id,
            "step_count": state["step_count"],
            "total_tokens": state["total_tokens"]}


# Routing
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END


#  Build the graph
graph = StateGraph(AgentState)
graph.add_node("agent",  agent_node)
graph.add_node("tools",  observable_tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

agent = graph.compile()

print("LangGraph agent with observability hooks compiled ")


LangGraph agent with observability hooks compiled 


## 5. Helper — `run_observed_agent`

In [6]:
def run_observed_agent(query: str) -> str:
    """
    Convenience wrapper that:
      1. Opens a trace
      2. Runs the LangGraph agent
      3. Closes the trace with timing + cost
      4. Returns the final answer string
    """
    trace_id = obs_logger.start_trace(query)

    initial_state: AgentState = {
        "messages":     [HumanMessage(content=query)],
        "trace_id":     trace_id,
        "step_count":   0,
        "total_tokens": 0,
    }

    try:
        final_state = agent.invoke(initial_state)
        answer = final_state["messages"][-1].content
        status = "success"
        total_tokens = final_state.get("total_tokens", 0)
    except Exception as e:
        answer = f"Agent error: {e}"
        status = "error"
        total_tokens = 0

    obs_logger.end_trace(trace_id, answer,
                         status=status, total_tokens=total_tokens)

    return answer, trace_id

print("run_observed_agent helper ready ")


run_observed_agent helper ready 


## 6. Demo — Logging Agent Decisions

### Demo 1 — Single-tool query (simple reasoning)

Ask about one suspect's alibi. We expect a **single `check_alibi` tool call** followed by a short reasoning chain.


In [7]:
answer_1, trace_id_1 = run_observed_agent(
    "Check the alibi of Butler Grey for the night the Rosewood Emerald was stolen."
)

print("\nFINAL ANSWER:", answer_1)


/tmp/ipykernel_1460/3146045309.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),



    NEW TRACE STARTED  [bb88e02e]
  Query: Check the alibi of Butler Grey for the night the Rosewood Emerald was stolen.


/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'check_alibi' with args {'suspect': 'Butler Grey'}
  🔧 [TOOL_CALL   ] check_alibi({'suspect': 'Butler Grey'})
  📦 [TOOL_RESULT ] BUTLER_GREY: "Says he was polishing silverware in the dining room."  ✅ Verified
              ⏱  53 ms
  🧠 [REASONING   ] Butler Grey's alibi for the night the Rosewood Emerald was stolen is that he was polishing silverware in the dining room
              ⏱  3374 ms

  TRACE COMPLETE  [bb88e02e]  status=success
  Total time : 5679 ms
  Est. cost  : $0.000084


FINAL ANSWER: Butler Grey's alibi for the night the Rosewood Emerald was stolen is that he was polishing silverware in the dining room, and this alibi has been verified.


/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1460/3146045309.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


### Demo 2 — Multi-tool reasoning (alibi + evidence + metric)

Ask a compound question that forces the agent to call **multiple tools** in sequence.


In [8]:
answer_2, trace_id_2 = run_observed_agent(
    "Investigate Chef Dubois in the theft of the Rosewood Emerald: check his alibi, "
    "search the kitchen for evidence, then calculate his guilt score assuming an alibi strength of 0.3."
)

print("\nFINAL ANSWER:", answer_2)


/tmp/ipykernel_1460/3146045309.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),



    NEW TRACE STARTED  [90720587]
  Query: Investigate Chef Dubois in the theft of the Rosewood Emerald: check his alibi, s...


/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'check_alibi' with args {'suspect': 'Chef Dubois'}
  ⚡ [DECISION    ] Calling tool 'search_crime_scene_evidence' with args {'location': 'kitchen'}
  ⚡ [DECISION    ] Calling tool 'calculate_case_metric' with args {'metric': 'guilt_score', 'evidence_count': 0, 'alibi_strength': 0.3, 'su
  🔧 [TOOL_CALL   ] check_alibi({'suspect': 'Chef Dubois'})
  🔧 [TOOL_CALL   ] search_crime_scene_evidence({'location': 'kitchen'})
  🔧 [TOOL_CALL   ] calculate_case_metric({'metric': 'guilt_score', 'evidence_count': 0, 'alibi_strength': 0.3, 'suspects_interviewed_count'
  📦 [TOOL_RESULT ] CHEF_DUBOIS: "Claims he was preparing dinner in the kitchen all evening."  ⚠️ Unverified
              ⏱  85 ms
  📦 [TOOL_RESULT ] • Knife set slightly out of order.
• A receipt for a locksmith, dated last week.
              ⏱  85 ms
  📦 [TOOL_RESULT ] Guilt Score = 0.00
              ⏱  85 ms
  🧠 [REASONING   ] Here's the investigation summary for Chef Dubois regarding the theft of the 

/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1460/3146045309.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


### Demo 3 — Comparative analysis (multiple suspects)

Force the agent to retrieve data for **two suspects** and synthesise a comparison.


In [9]:
answer_3, trace_id_3 = run_observed_agent(
    "Compare Lady Sinha and Gardener Hayes as suspects: check their alibis, search the "
    "gallery and greenhouse for evidence, then calculate the case strength for each "
    "assuming 4 suspects interviewed so far for Sinha and 6 for Hayes."
)

print("\nFINAL ANSWER:", answer_3)


/tmp/ipykernel_1460/3146045309.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),



    NEW TRACE STARTED  [98efcba2]
  Query: Compare Lady Sinha and Gardener Hayes as suspects: check their alibis, search th...


/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'check_alibi' with args {'suspect': 'Lady Sinha'}
  ⚡ [DECISION    ] Calling tool 'check_alibi' with args {'suspect': 'Gardener Hayes'}
  ⚡ [DECISION    ] Calling tool 'search_crime_scene_evidence' with args {'location': 'gallery'}
  ⚡ [DECISION    ] Calling tool 'search_crime_scene_evidence' with args {'location': 'greenhouse'}
  🔧 [TOOL_CALL   ] check_alibi({'suspect': 'Lady Sinha'})
  🔧 [TOOL_CALL   ] check_alibi({'suspect': 'Gardener Hayes'})
  🔧 [TOOL_CALL   ] search_crime_scene_evidence({'location': 'gallery'})
  🔧 [TOOL_CALL   ] search_crime_scene_evidence({'location': 'greenhouse'})
  📦 [TOOL_RESULT ] LADY_SINHA: "States she was reading in the west wing library."  ⚠️ Unverified
              ⏱  86 ms
  📦 [TOOL_RESULT ] GARDENER_HAYES: "Insists he was locking up the greenhouse at the time."  ✅ Verified
              ⏱  86 ms
  📦 [TOOL_RESULT ] • Faint fingerprints on the display case glass.
• A guest book missing its last page.
• Security camera u

/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'calculate_case_metric' with args {'metric': 'case_strength', 'evidence_count': 3, 'suspects_interviewed_co
  ⚡ [DECISION    ] Calling tool 'calculate_case_metric' with args {'metric': 'case_strength', 'evidence_count': 3, 'suspects_interviewed_co
  🔧 [TOOL_CALL   ] calculate_case_metric({'metric': 'case_strength', 'evidence_count': 3, 'suspects_interviewed_count': 4})
  🔧 [TOOL_CALL   ] calculate_case_metric({'metric': 'case_strength', 'evidence_count': 3, 'suspects_interviewed_count': 6})
  📦 [TOOL_RESULT ] Case Strength = 12 points  (0.1% case file complete)
              ⏱  34 ms
  📦 [TOOL_RESULT ] Case Strength = 18 points  (0.2% case file complete)
              ⏱  34 ms
  🧠 [REASONING   ] ### Comparison of Suspects: Lady Sinha vs. Gardener Hayes

#### Alibis:
- **Lady Sinha**: Claims she was reading in the 
              ⏱  3200 ms

  TRACE COMPLETE  [98efcba2]  status=success
  Total time : 6342 ms
  Est. cost  : $0.000267


FINAL ANSWER: ### Com

/tmp/ipykernel_1460/3146045309.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1460/3146045309.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


## 7. Execution Trace Inspection

Print the full, step-by-step trace for any of the runs above.  
Each row shows: **step number | step type | latency | content preview**.


In [10]:
#  Detailed report for Demo 2 (most interesting — multi-tool)
print("\n>>> Full trace for Demo 2 (Chef Dubois investigation):\n")
obs_logger.print_full_report(trace_id_2)



>>> Full trace for Demo 2 (Chef Dubois investigation):


#################################################################
  EXECUTION TRACE REPORT — [90720587]
#################################################################
  Query        : Investigate Chef Dubois in the theft of the Rosewood Emerald: check his alibi, search the kitchen for evidence, then calculate his guilt score assuming an alibi strength of 0.3.
  Status       : success
  Started      : 2026-09-29T20:53:24.778401
  Finished     : 2026-09-29T20:53:28.499461
  Total time   : 3721 ms
  Est. cost    : $0.000141
  Steps logged : 10

  Step 01 | DECISION     |      0 ms | Calling tool 'check_alibi' with args {'suspect': 'Chef Dubois'}
  Step 02 | DECISION     |      0 ms | Calling tool 'search_crime_scene_evidence' with args {'location': 'kitchen'}
  Step 03 | DECISION     |      0 ms | Calling tool 'calculate_case_metric' with args {'metric': 'guilt_score', 'evidence_count':
  Step 04 | TOOL_CALL    |      0 ms | che

In [11]:
# Detailed report for Demo 3 (comparison query)
print("\n>>> Full trace for Demo 3 (Sinha vs Hayes comparison):\n")
obs_logger.print_full_report(trace_id_3)



>>> Full trace for Demo 3 (Sinha vs Hayes comparison):


#################################################################
  EXECUTION TRACE REPORT — [98efcba2]
#################################################################
  Query        : Compare Lady Sinha and Gardener Hayes as suspects: check their alibis, search the gallery and greenhouse for evidence, then calculate the case strength for each assuming 4 suspects interviewed so far for Sinha and 6 for Hayes.
  Status       : success
  Started      : 2026-09-29T20:53:38.713441
  Finished     : 2026-09-29T20:53:45.055057
  Total time   : 6342 ms
  Est. cost    : $0.000267
  Steps logged : 19

  Step 01 | DECISION     |      0 ms | Calling tool 'check_alibi' with args {'suspect': 'Lady Sinha'}
  Step 02 | DECISION     |      0 ms | Calling tool 'check_alibi' with args {'suspect': 'Gardener Hayes'}
  Step 03 | DECISION     |      0 ms | Calling tool 'search_crime_scene_evidence' with args {'location': 'gallery'}
  Step 04 | DECISI

## 8. Observability Dashboard — Cost & Latency

The dashboard aggregates **all traces** in one view — useful for identifying:
- Which queries are slowest (latency hotspots)
- Which runs used the most steps (potential infinite-loop risk)
- Cost trends across different query types


In [12]:
obs_logger.print_summary_dashboard()



  📊  OBSERVABILITY DASHBOARD — ALL TRACES
  Trace    Status   Steps  Time(ms)   Est.Cost    Query
  ------------------------------------------------------------
  bb88e02e success       4       5679 $0.000084  Check the alibi of Butler Grey for ...
  90720587 success      10       3721 $0.000141  Investigate Chef Dubois in the thef...
  98efcba2 success      19       6342 $0.000267  Compare Lady Sinha and Gardener Hay...



### Per-step latency breakdown (Demo 2)

Drill into which **step type** took the most time.


In [13]:
trace = obs_logger.traces[trace_id_2]

step_latencies: Dict[str, List[float]] = {}
for step in trace.steps:
    step_latencies.setdefault(step.step_type, []).append(step.latency_ms)

print(f"\nLatency breakdown for trace [{trace_id_2}]:")
print(f"{'Step Type':15s}  {'Count':6s}  {'Total ms':10s}  {'Avg ms':8s}")
print("-" * 45)
for stype, latencies in sorted(step_latencies.items()):
    total = sum(latencies)
    avg   = total / len(latencies)
    print(f"  {stype:13s}  {len(latencies):6d}  {total:10.1f}  {avg:8.1f}")
print(f"\n  Total wall-clock: {trace.total_ms:.0f} ms")



Latency breakdown for trace [90720587]:
Step Type        Count   Total ms    Avg ms  
---------------------------------------------
  decision            3         0.0       0.0
  reasoning           1      2078.8    2078.8
  tool_call           3         0.0       0.0
  tool_result         3       254.5      84.8

  Total wall-clock: 3721 ms


## 9. Exporting Traces (JSON)

In production you would ship traces to an external store (Elastic, BigQuery, S3 …).  
Below we serialise a trace to JSON — ready to ingest into any logging pipeline.


In [14]:
import dataclasses

def trace_to_dict(trace: ExecutionTrace) -> dict:
    """Convert an ExecutionTrace to a plain dict for JSON export."""
    return {
        "trace_id":       trace.trace_id,
        "query":          trace.query,
        "status":         trace.status,
        "started_at":     trace.started_at,
        "finished_at":    trace.finished_at,
        "total_ms":       trace.total_ms,
        "estimated_cost": trace.estimated_cost,
        "final_answer":   trace.final_answer,
        "steps": [
            {
                "step_id":    s.step_id,
                "step_type":  s.step_type,
                "timestamp":  s.timestamp,
                "content":    str(s.content)[:300],
                "latency_ms": s.latency_ms,
                "metadata":   s.metadata,
            }
            for s in trace.steps
        ],
    }

trace_json = json.dumps(trace_to_dict(obs_logger.traces[trace_id_2]), indent=2)
print("Sample JSON export (first 1500 chars):")
print(trace_json[:1500])
print("\n... (truncated for display)")


Sample JSON export (first 1500 chars):
{
  "trace_id": "90720587",
  "query": "Investigate Chef Dubois in the theft of the Rosewood Emerald: check his alibi, search the kitchen for evidence, then calculate his guilt score assuming an alibi strength of 0.3.",
  "status": "success",
  "started_at": "2026-09-29T20:53:24.778401",
  "finished_at": "2026-09-29T20:53:28.499461",
  "total_ms": 3721.06,
  "estimated_cost": 0.00014145,
  "final_answer": "Here's the investigation summary for Chef Dubois regarding the theft of the Rosewood Emerald:\n\n1. **Alibi Check**: Chef Dubois claims he was preparing dinner in the kitchen all evening. However, this alibi is unverified.\n\n2. **Evidence Found in the Kitchen**:\n   - A knife set was found slightly out of order.\n   - A receipt for a locksmith, dated last week, was also discovered.\n\n3. **Guilt Score Calculation**: Based on the evidence count (0 pieces directly tying him to the scene), an alibi strength of 0.3, and having interviewed 1 suspect

## 10. Reflection Questions

1. **Why is observability considered a first-class concern** in production agent systems, not just a debugging afterthought?

2. **How do execution traces differ from traditional application logs?** What extra information do they provide and when does that matter?

3. The `decision` step logs the agent's intent *before* a tool is called, while `tool_result` logs the output *after*. **Why is logging both sides important?** Give a concrete debugging scenario where one side alone would be insufficient.

4. Looking at the latency breakdown in Section 8 — **which step type tends to dominate?** What strategies would you use to reduce end-to-end latency?

5. In this demo cost estimation is very rough (token count × fixed rate). **What additional factors should a production cost tracker capture?**

6. **How would you extend this observability layer to detect anomalies automatically** — for example, a tool call loop or an unexpectedly high step count?

---

**Key Takeaway:**  
Observability transforms an agent from a black box into a transparent system.  
Every decision, tool call, and reasoning step can be captured, inspected, and acted upon — enabling confident debugging, cost control, and compliance auditing in production LangGraph deployments.
